# Cowpea IITA — PySpark

This notebook analyses the cowpea dataset using **PySpark only** (no pandas). Intended for use on Databricks.

| | |
| --- | --- |
| **Crop** | Cowpea (_Vigna unguiculata_) |
| **Locations** | Ibadan, Ikenne, Kano (Nigeria) |
| **Years** | 2020–2022 |
| **Measurements** | 3,360 |
| **Genotypes** | 112 (98 with SNP markers) |
| **Tables** | `cowpea_iita_measurements`, `cowpea_iita_snp` |

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.connect.session import SparkSession

CATALOG = "open_jii_data_hackathon.default"

spark = SparkSession.getActiveSession()
assert spark is not None, "This notebook must be run on Databricks"

ms = spark.table(f"{CATALOG}.cowpea_iita_measurements")
snp = spark.table(f"{CATALOG}.cowpea_iita_snp")

In [ ]:
ms.printSchema()
ms.show(5)

In [ ]:
snp.printSchema()
snp.show(5, truncate=40)

## EDA

In [ ]:
print(f"Measurements: {ms.count():,}")
print(f"SNP markers: {snp.count():,}")
print(f"Genotypes in measurements: {ms.select('genotype').distinct().count()}")
print(f"Environments: {ms.select('env_id').distinct().count()}")

In [ ]:
# Counts by environment
ms.groupBy("env_id", "location", "year", "stress_condition") \
  .count() \
  .orderBy("env_id") \
  .show()

In [ ]:
# Summary statistics for phenotype columns
phenotype_cols = [
    "relative_chlorophyll_content",
    "leaf_angle",
    "leaf_temp_differential",
    "lef",
    "npqt",
    "phi2",
    "phi_no",
    "phi_npq",
]

ms.select(phenotype_cols).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

In [ ]:
# Mean phenotype values by stress condition
ms.groupBy("stress_condition") \
  .agg(
      F.mean("phi2").alias("mean_phi2"),
      F.mean("lef").alias("mean_lef"),
      F.mean("npqt").alias("mean_npqt"),
      F.mean("relative_chlorophyll_content").alias("mean_spad"),
      F.mean("leaf_temp_differential").alias("mean_leaf_temp_diff"),
  ) \
  .show()

In [ ]:
# Mean phenotype values by location and stress
ms.groupBy("location", "stress_condition") \
  .agg(
      F.count("*").alias("n"),
      F.mean("phi2").alias("mean_phi2"),
      F.stddev("phi2").alias("std_phi2"),
      F.mean("lef").alias("mean_lef"),
      F.mean("npqt").alias("mean_npqt"),
  ) \
  .orderBy("location", "stress_condition") \
  .show()

In [ ]:
# Top 10 genotypes by mean Phi2
ms.groupBy("genotype") \
  .agg(
      F.count("*").alias("n"),
      F.mean("phi2").alias("mean_phi2"),
      F.mean("lef").alias("mean_lef"),
      F.mean("npqt").alias("mean_npqt"),
  ) \
  .orderBy(F.desc("mean_phi2")) \
  .show(10)

## Correlations

In [ ]:
# Pairwise correlations between phenotype traits
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=phenotype_cols, outputCol="features", handleInvalid="skip")
vec_df = assembler.transform(ms)

corr_matrix = Correlation.corr(vec_df, "features", "pearson").head()[0]
print("Correlation matrix:")
for i, col in enumerate(phenotype_cols):
    vals = " ".join(f"{corr_matrix[i, j]:6.2f}" for j in range(len(phenotype_cols)))
    print(f"{col:>32s}  {vals}")

## SNP data exploration

In [ ]:
# Markers per chromosome
snp.groupBy("chromosome") \
  .count() \
  .orderBy("chromosome") \
  .show()

In [ ]:
# Example: join mean phenotypes with SNP data
# Unpivot SNP table to long format for a specific marker
genotype_cols = [c for c in snp.columns if c.startswith("G")]

genotype_means = ms.groupBy("genotype") \
    .agg(F.mean("phi2").alias("mean_phi2"))

# Pick one marker as example
first_marker = snp.limit(1)
marker_id = first_marker.select("marker_id").head()[0]

# Unpivot genotype columns to rows
stack_expr = ", ".join(f"'{c}', `{c}`" for c in genotype_cols)
snp_long = first_marker.selectExpr(
    "marker_id",
    f"stack({len(genotype_cols)}, {stack_expr}) as (genotype, allele)",
)

# Join with phenotype means
joined = snp_long.join(genotype_means, on="genotype")

print(f"Marker: {marker_id}")
joined.groupBy("allele") \
  .agg(F.count("*").alias("n"), F.mean("mean_phi2").alias("phi2_by_allele")) \
  .show()